In [1]:
!pip install lxml geopy

  Obtaining dependency information for geopy from https://files.pythonhosted.org/packages/66/e5/de8b557b279e31cdfb2070ac83eb45dc7e29bd93cd544128756f658e52c7/geopy-2.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for geographiclib<3,>=1.52 from https://files.pythonhosted.org/packages/31/b3/802576f2ea5dcb48501bb162e4c7b7b3ca5654a42b2c968ef98a797a4c79/geographiclib-2.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
   --- ------------------------------------ 10.2/114.6 kB ? eta -:--:--
   ------------- ------------------------- 41.0/114.6 kB 393.8 kB/s eta 0:00:01
   -------------------------------------- 114.6/114.6 kB 835.0 kB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 40.7/40.7 kB 2.0 MB/s eta 0:00:00


In [4]:
import os
import glob
from lxml import etree
from geopy.geocoders import Nominatim
import time

# 1. Geocoder initialisieren (User-Agent ist für Nominatim Pflicht)
geolocator = Nominatim(user_agent="tei_place_extractor_script")

# 2. Namespace-Handling für TEI
# TEI-Dateien nutzen meist den standardmäßigen TEI-Namespace
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

# Verzeichnis festlegen
folder_path = 'tei'

# Dictionary zur Speicherung der eindeutigen Orte
# Key: Ort (String), Value: Set von Typen (z. B. 'Absendeort', 'Empfangsort')
places_dict = {}

print("Lese XML-Dateien aus...")

# 3. XML-Dateien durchsuchen mit einem "toleranten" Parser
xml_files = glob.glob(os.path.join(folder_path, '*.xml'))

# Erstelle einen Parser, der Ungereimtheiten/Schema-Fehler ignoriert
parser = etree.XMLParser(recover=True)

for file_path in xml_files:
    try:
        # Den toleranten Parser beim Einlesen übergeben
        tree = etree.parse(file_path, parser)
        root = tree.getroot()
        
        # Suchen nach <placeName ref="..."> sowohl mit als auch ohne TEI-Namespace
        place_names = root.xpath('//tei:placeName[@ref] | //placeName[@ref]', namespaces=NS)
        
        for elem in place_names:
            name = elem.text.strip() if elem.text else None
            note_type = elem.get('type') or elem.get('role') or "MentionedPlace"
            
            if name:
                if name not in places_dict:
                    places_dict[name] = set()
                places_dict[name].add(note_type)
                
    except Exception as e:
        print(f"Fehler beim Lesen von {file_path}: {e}")

# 4. Erstellen der TEI <standOff>-Struktur
standOff = etree.Element("standOff")
listPlace = etree.SubElement(standOff, "listPlace")

# Cache, um Mehrfach-Anfragen an den Geocoder zu vermeiden
geo_cache = {}

print("\nHole Koordinaten und erstelle <standOff>...")

for place_name, roles in places_dict.items():
    # erstelle eine saubere xml:id aus dem Ortsnamen (Kleinschreibung, Leerzeichen durch Unterstrich)
    clean_id = place_name.lower().replace('å', 'a')
    
    place_elem = etree.SubElement(listPlace, "place", {etree.QName("http://www.w3.org/XML/1998/namespace", "id"): clean_id})
    
    # <placeName>
    p_name = etree.SubElement(place_elem, "placeName")
    p_name.text = place_name
    
    # <note> für jede gefundene Rolle
    for role in roles:
        note = etree.SubElement(place_elem, "note", type="typePlace")
        note.text = role
        
    # Geocoding / Koordinaten abrufen
    if place_name not in geo_cache:
        try:
            # Höfliche Pause von 1 Sekunde für den kostenlosen Nominatim-Dienst
            time.sleep(1)
            location = geolocator.geocode(place_name)
            if location:
                geo_cache[place_name] = f"{location.latitude},{location.longitude}"
            else:
                geo_cache[place_name] = "N/A"
        except Exception as e:
            print(f"Geocoding-Fehler bei {place_name}: {e}")
            geo_cache[place_name] = "N/A"
            
    coords = geo_cache[place_name]
    
    # <location><geo>
    loc_elem = etree.SubElement(place_elem, "location")
    geo_elem = etree.SubElement(loc_elem, "geo")
    geo_elem.text = coords

# 5. Ergänztes Ergebnis als XML speichern
output_tree = etree.ElementTree(standOff)
output_file = "standOff_result.xml"

output_tree.write(
    output_file, 
    pretty_print=True, 
    xml_declaration=True, 
    encoding="UTF-8"
)

print(f"\nFertig! Das <standOff> wurde erfolgreich in '{output_file}' gespeichert.")

Lese XML-Dateien aus...

Hole Koordinaten und erstelle <standOff>...

Fertig! Das <standOff> wurde erfolgreich in 'standOff_result.xml' gespeichert.


In [1]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict

# TEI-Namensraum
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

# Pfad zu deinen XML-Editionen
EDITIONS_DIR = 'tei'

# Datenstruktur: { "skane": {"employment": 5, "application": 7, "residence": 0} }
place_counts = defaultdict(lambda: defaultdict(int))

# Alle XML-Dateien im Ordner durchsuchen
for filename in os.listdir(EDITIONS_DIR):
    if filename.endswith('.xml'):
        filepath = os.path.join(EDITIONS_DIR, filename)
        
        try:
            tree = ET.parse(filepath)
            root = tree.getroot()
            
            # Alle placeName-Elemente mit ref-Attribut finden
            for place in root.findall('.//tei:placeName[@ref]', NS):
                ref = place.attrib.get('ref', '').replace('#', '').strip()
                if not ref:
                    continue
                
                # Typ bestimmen: Sucht zuerst nach type="..." am placeName
                place_type = place.attrib.get('type')
                
                # Falls nicht vorhanden, schauen wir auf das Elternelement (z. B. <residence>)
                if not place_type:
                    # Alternativ im Kontext suchen
                    parent_tag = place.find('..').tag.replace(f"{{{NS['tei']}}}", '')
                    if parent_tag in ['residence', 'employment', 'application']:
                        place_type = parent_tag
                    else:
                        place_type = 'unspecified' # Falls kein Typ angegeben ist
                
                # Zähler erhöhen
                place_counts[ref][place_type] += 1
                
        except Exception as e:
            print(f"Fehler beim Verarbeiten von {filename}: {e}")

# Ergebnis in der Konsole ausgeben
print("Gezählte Orts-Häufigkeiten:")
for place_id, counts in place_counts.items():
    print(f"{place_id}: " + ", ".join([f"{k}: {v}" for k, v in counts.items()]))

Fehler beim Verarbeiten von 1906_10_27_SD_PS_002.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_10_28_SD_PS_003.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_10_29_SD_PS_001.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_11_04_SD_PS_001.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_11_14_SD_PS_001.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_11_15_SD_PS_001.xml: 'NoneType' object has no attribute 'tag'
Fehler beim Verarbeiten von 1906_11_18_SD_PS_001.xml: 'NoneType' object has no attribute 'tag'
Gezählte Orts-Häufigkeiten:
perstorp: application: 2
skane: application: 4, employment: 28
engelholm: application: 2
stockholm: application: 32, residence: 1, employment: 2
tagarp: application: 4, employment: 2
malmö: application: 128, employment: 33, residence: 2
trolleholm: employment: 3
sjöbo: employment: 4
hammarskjöld: applicati